# Week 1, Lab 3 — Tools from scratch

No SDK. You describe tools in the prompt, parse the model's intent, run Python, send the result back.

This is what OpenAI Agents SDK, CrewAI, and LangChain are doing under the hood.


## 1. Setup


In [ ]:
WEEK = 'Week 1'
LAB = 'Lab 3 — tools from scratch'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


In [ ]:
if BACKEND == "huggingface":
    %pip install -q transformers torch accelerate fastapi uvicorn pydantic
else:
    %pip install -q ollama pydantic


## 2. Tools are just functions + a JSON schema


In [ ]:
print("calculator(2+2) =", calculator("2+2"))
print("lookup_fact(mcp) =", lookup_fact("mcp"))
print("today_date() =", today_date())
print("schemas:", [t["name"] for t in TOOL_SCHEMAS])


## 3. Teach the model the tool format


In [ ]:
TOOLS_PREAMBLE = """You are a tool-using assistant.
If you need a tool, reply with ONLY JSON:
{"name": "<tool>", "arguments": {<args>}}
If you can answer without a tool, reply with normal text.

Tools:
- calculator(expression: str) — arithmetic only
- lookup_fact(topic: str) — local facts about agentic AI
- today_date() — today's date
"""

def run_one_step(user_text: str) -> str:
    reply = local_chat(
        [
            {"role": "system", "content": TOOLS_PREAMBLE},
            {"role": "user", "content": user_text},
        ],
        max_new_tokens=120,
        temperature=0.1,
    )
    print("MODEL:", reply)
    call = parse_tool_call(reply)
    if not call:
        return reply
    fn = {"calculator": calculator, "lookup_fact": lookup_fact, "today_date": today_date}.get(call["name"])
    if fn is None:
        return f"Unknown tool: {call['name']}"
    result = fn(**call["arguments"]) if call["arguments"] else fn()
    print("TOOL", call["name"], "->", result)
    final = local_chat(
        [
            {"role": "system", "content": "Answer the user using the tool result. Be brief."},
            {"role": "user", "content": user_text},
            {"role": "assistant", "content": reply},
            {"role": "user", "content": f"TOOL RESULT: {result}"},
        ],
        max_new_tokens=120,
        temperature=0.2,
    )
    return final

print("\n=== math ===")
print("FINAL:", run_one_step("What is 45 * 12 + 30?"))
print("\n=== fact ===")
print("FINAL:", run_one_step("What is MCP in one sentence?"))


## 4. Exercise

1. Add a `reverse_text(text)` tool and ask the model to reverse a name.
2. Intentionally omit the JSON instruction and see how often tool-calling breaks.
3. Log `(thought?, tool, args, result)` as a list — that log is the ancestor of tracing in later SDKs.

**Next:** `lab4_simple_agent_loop.ipynb` — repeat Reason → Act → Observe until done.
